
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [40]:
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=ChatGroq(model="qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x70490d733650>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x70490d733e50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [22]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [26]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x704903bc9480>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x704903bc9d00>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 

In [19]:
model.invoke("Provide details about the moview Titanic")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Titanic. Let me start by recalling what I know about it. I remember it\'s a big Hollywood film from the 90s, directed by James Cameron. The main stars are Leonardo DiCaprio and Kate Winslet. The story is about the sinking of the RMS Titanic, which was a real historical event. \n\nFirst, I should confirm the release date. I think it came out in 1997. The director is definitely James Cameron. The main characters are Jack and Rose, played by DiCaprio and Winslet. The plot is a love story between a wealthy woman and a poor artist, set against the backdrop of the Titanic\'s maiden voyage. The ship hits an iceberg and sinks, leading to the tragic ending.\n\nI should mention the historical context. The Titanic was a British passenger liner that sank in 1912 after hitting an iceberg. The movie combines the real events with the fictional love story. The film was a massive success, both critically and commercially. It

In [30]:
response=model_with_structure.invoke("Provide details about the movie Titanic")
response

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8)

### Message output alongside parsed structure

In [31]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Titanic")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Titanic. Let me think about the tools provided. There\'s a Movie function that requires title, year, director, and rating. I need to recall the information about Titanic. The title is obviously "Titanic". The year it was released was 1997. The director is James Cameron. The rating, I think it\'s around 7.8 on IMDb, but since the rating is out of 10, maybe I should use that. Wait, the function parameters specify a number for the rating. Let me confirm: the user didn\'t specify which rating, but I\'ll go with the IMDb rating. So putting it all together, I need to structure the JSON object with those parameters. Make sure all required fields are included. Let me check the required fields again: title, year, director, rating. Yep, all there. Alright, I\'ll format the tool call accordingly.\n', 'tool_calls': [{'id': 'sr0sc23ak', 'function': {'arguments': '{"director":"

### Nested Structure

In [34]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Titanic")
response

MovieDetails(title='Titanic', year=1997, cast=[Actor(name='Leonardo DiCaprio', role='Jack'), Actor(name='Kate Winslet', role='Rose')], genres=['Romance', 'Disaster'], budget=200.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [35]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("Please provide the details of the movie Titanic")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Titanic', 'year': 1997}

In [37]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Titanic")
response

{'budget': 200000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jack'},
  {'name': 'Kate Winslet', 'role': 'Rose'}],
 'genres': ['Romance', 'Drama'],
 'title': 'Titanic',
 'year': 1997}

In [38]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [22]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [41]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

user_info = """
    Hey everyone, I am Md Ahsan Habib Sadik, to contract with me you can use ahsan234@gmail.com or 01334343434
"""

result = agent.invoke({
    "messages": [{"role": "user", "content": f"Extract the Contract info from {user_info}"}]
})

result

{'messages': [HumanMessage(content='Extract the Contract info from \n    Hey everyone, I am Md Ahsan Habib Sadik, to contract with me you can use ahsan234@gmail.com or 01334343434\n', additional_kwargs={}, response_metadata={}, id='1a2b2bb8-5e6a-4d9b-8b30-c59ac708cd74'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact info from the given message. The message says: "Hey everyone, I am Md Ahsan Habib Sadik, to contract with me you can use ahsan234@gmail.com or 01334343434". \n\nFirst, I need to identify the name. The person introduces themselves as Md Ahsan Habib Sadik. So the name parameter should be "Md Ahsan Habib Sadik".\n\nNext, the email. The message mentions "ahsan234@gmail.com". That\'s clearly an email address, so the email parameter is that.\n\nThen the phone number is "01334343434". The function requires phone number, so that\'s the phone parameter.\n\nI need to make sure all required fields are included. 

In [24]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### TypedDict

In [42]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [43]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')